# Indic Voice Pipeline — ASR evaluation on Colab GPU

Runs the evaluation harness on a T4: base Whisper vs your LoRA adapter, on
identical audio, through the explicit `ASRRunner` (no `generate()`, no
`pipeline("asr")`).

**Runtime → Change runtime type → T4 GPU** before running anything.

Per run this produces `run_config.json` (git SHA, versions, device, exact
sampled indices), `predictions.jsonl`, `metrics.json` (WER/CER at four
normalization levels, error categories, latency) and `errors.jsonl`.

Google Drive checkpoints are treated as **read-only** throughout.

## 0. Is training still running?

**Do not run this notebook on the GPU that is training.** Two things go wrong:

1. **Latency becomes fiction.** RTF, p50 and p90 measured while a training job
   saturates the same SMs describe contention, not your serving path. `--warmup`
   does not help — it removes cold-start cost, not a competing process.
2. **You may OOM.** whisper-medium LoRA training with `predict_with_generate`
   is already tight on a 16 GB T4. A second whisper-medium for evaluation can
   push it over, and the OOM may land in the training process.

Use a **separate Colab runtime**, or wait for training to finish. If you must
measure WER now on the same box, pass
`--note "GPU shared with training run - latency invalid"` so the caveat is
recorded in `run_config.json` rather than remembered.

In [ ]:
!nvidia-smi

If the memory column already shows several GB in use by another process, stop
here and open a second runtime.

## 1. Clone and install

`scripts/setup.sh` pins `datasets<4`. This is load-bearing: `google/fleurs` is
still a script-backed dataset, and `datasets>=4.0` removed loading-script
support entirely, so an unpinned install fails on every FLEURS load.

In [ ]:
!git clone https://github.com/Vaibhav7711/indic-voice-pipeline.git
%cd indic-voice-pipeline
!bash scripts/setup.sh

If `datasets` was already imported before the pin took effect,
**Runtime → Restart session**, then resume from the next cell (not the install).

In [ ]:
%cd /content/indic-voice-pipeline
!python scripts/preflight.py

## 2. Sanity-check the scoring layer

CPU-only, under a second. Verifies the metric against an independent
Levenshtein implementation and checks the Devanagari normalization rules. Run
before spending GPU time.

In [ ]:
!python -m pytest tests/ -q

## 3. Find the checkpoint

Do **not** hardcode a step number. `save_total_limit` rotates old checkpoints
away as training advances, so `checkpoint-1400` stops existing once training
passes it. `load_best_model_at_end=True` protects the best checkpoint from
rotation, so what survives is roughly "the best one plus the most recent ones".

`resolve_adapter` takes a policy instead:

| Policy | Meaning |
| --- | --- |
| `latest` | Highest step number. Newest, not necessarily best. |
| `best` | What the Trainer recorded as best by validation WER. |
| `final` | The exported `best/` directory, written only when training completes. |

In [ ]:
from google.colab import drive
drive.mount('/content/drive', readonly=True)   # read-only: never writes to whisper-training/

from benchmarks.checkpoints import list_checkpoints, resolve_adapter, verify_adapter

TRAIN_DIR = "/content/drive/MyDrive/whisper-training"

for step, path in list_checkpoints(TRAIN_DIR):
    print(f"  {step:>6}  {path.name}")

In [ ]:
# 'best' is the right default for reporting. Fall back to 'latest' while
# training is mid-flight and the recorded best may have been rotated away.
from benchmarks.checkpoints import CheckpointError

try:
    ADAPTER_SRC = resolve_adapter(TRAIN_DIR, "best")
    POLICY = "best"
except CheckpointError as exc:
    print("best unavailable:", exc, "\n-> falling back to latest")
    ADAPTER_SRC = resolve_adapter(TRAIN_DIR, "latest")
    POLICY = "latest"

print(f"\nusing ({POLICY}): {ADAPTER_SRC}")

### Copy it off Drive before loading

Two reasons. Drive FUSE reads are slow, and re-reading weights every run adds
load to the same mount the training job is writing checkpoints to.

`stage_checkpoint` also verifies the safetensors header against the real file
size. If the Trainer is mid-save on that checkpoint, the file is truncated and
this raises immediately rather than failing later inside safetensors. If it
does raise, wait a minute and use the previous checkpoint.

Only inference files are copied — `optimizer.pt`, `scheduler.pt` and RNG state
are resume artefacts worth hundreds of megabytes. Nothing is written back to
the source.

In [ ]:
from benchmarks.checkpoints import stage_checkpoint

ADAPTER = str(stage_checkpoint(ADAPTER_SRC, "/content/adapters"))
info = verify_adapter(ADAPTER)
print(ADAPTER)
for k in ("r", "lora_alpha", "target_modules", "base_model", "has_tokenizer"):
    print(f"  {k:<16} {info[k]}")

`has_tokenizer: False` is expected for an intermediate `checkpoint-*` folder —
`loader.py` falls back to the base Whisper processor in that case. The exported
`best/` directory does contain tokenizer files.

## 4. Configuration

`SEED` and `LIMIT` must be identical across both runs or the comparison is
meaningless — `benchmarks/compare.py` refuses to report a delta otherwise.

**While training is still running, use `SPLIT = "validation"`.** The test split
is for final reporting only; every look at it erodes its independence, and the
model is not converged yet.

Start with `LIMIT = 100` to confirm the pipeline end to end. FLEURS Hindi test
is roughly 400 examples. To use a whole split, delete the `--limit {LIMIT}`
flag from the run cells rather than setting `LIMIT = None` — shell
interpolation would pass the literal string `None` to argparse.

In [ ]:
MODEL = "openai/whisper-medium"
SPLIT = "validation"    # switch to "test" only for the final, post-training number
LIMIT = 100
SEED  = 0
LEVEL = "standard"
NOTE  = ""              # e.g. "GPU shared with training run - latency invalid"

tag = f"{SPLIT}-{LIMIT}"
BASE_DIR = f"results/eval/medium-base-{tag}"
LORA_DIR = f"results/eval/medium-{POLICY}-{tag}"
print(BASE_DIR, "\n", LORA_DIR)

## 5. Baseline: whisper-medium, no adapter

The first FLEURS load downloads and prepares the split — slow once, cached
after. `--warmup 2` discards two GPU passes before timing, because the first
forward pays for cuDNN autotuning and allocator growth and would otherwise
poison the mean RTF.

In [ ]:
!python -m benchmarks.asr_eval run \
    --model {MODEL} \
    --split {SPLIT} --limit {LIMIT} --seed {SEED} \
    --level {LEVEL} --warmup 2 --note "{NOTE}" \
    --out-dir {BASE_DIR}

## 6. Candidate: same audio, with the LoRA adapter merged

In [ ]:
!python -m benchmarks.asr_eval run \
    --model {MODEL} --adapter {ADAPTER} \
    --split {SPLIT} --limit {LIMIT} --seed {SEED} \
    --level {LEVEL} --warmup 2 --note "{NOTE}" \
    --out-dir {LORA_DIR}

## 7. Compare

This refuses to print a delta unless both runs used the same split, the same
example indices and the same normalization level. That guard is the point —
comparing a fine-tuned model on one sample against a baseline on another is the
easiest way to manufacture an improvement that is not real.

In [ ]:
!python -m benchmarks.compare \
    --baseline {BASE_DIR} \
    --candidate {LORA_DIR} \
    --out results/eval/compare-base-vs-{POLICY}.json

## 8. Read the failure modes

The error-category table says *what to fix next*:

| Dominant category | What it points at |
| --- | --- |
| `truncation` / `hallucination` | Decoding: `max_new_tokens`, early EOS, repetition |
| `orthographic` | Not a modelling error — a normalization policy decision |
| `code_switch` / `rare_word` | Training data: Hinglish, named entities |
| `numeric` | Output-format policy (digits vs words) |
| `function_word` | Usually audio quality, not vocabulary |

In [ ]:
import json
m = json.load(open(f"{LORA_DIR}/metrics.json"))

print("WER %", m["headline"]["wer_percent"], " CER %", m["headline"]["cer_percent"])
print("\nNormalization sensitivity:", json.dumps(m["normalization_sensitivity"], indent=2))
print("\nError categories:")
for name, payload in m["error_analysis"]["by_category"].items():
    print(f"  {name:<16}{payload['errors']:>6}{payload['share_of_errors_percent']:>8}%")
print("\nFlags:", m["error_analysis"]["flag_counts"])

### Worst examples — read these before changing anything

In [ ]:
for ex in m["error_analysis"]["worst_examples"][:8]:
    print(f"[{ex['id']}] WER {ex['wer']:.2f}")
    print("  ref:", ex["reference"])
    print("  hyp:", ex["hypothesis"])
    print()

### Top confusion pairs per category

In [ ]:
for name, payload in m["error_analysis"]["by_category"].items():
    pairs = payload["top_confusions"][:5]
    if pairs:
        print(f"{name}:")
        for pair, count in pairs:
            print(f"   {count:>3}x  {pair}")

## 9. Re-score without the GPU

Scoring has no torch dependency, so you can change the normalization level or
the structural-run threshold without re-running inference. Useful for asking
"how much of this WER is spelling convention?"

In [ ]:
!python -m benchmarks.asr_eval score \
    --predictions {LORA_DIR}/predictions.jsonl \
    --level aggressive \
    --out-dir {LORA_DIR}-aggressive

## 10. Seed the hard set from real failures

Writes `status="candidate"` items only. They are excluded from every reported
metric until you listen to each clip, correct the transcript, assign categories
and flip the status to `curated`. See `data/hard_set/README.md`.

Some FLEURS references are simply wrong — those are bad references, not hard
audio. Delete them during review rather than promoting them.

In [ ]:
!python -m benchmarks.hard_set bootstrap \
    --predictions {LORA_DIR}/predictions.jsonl \
    --out data/hard_set/candidates.jsonl \
    --limit 30 --min-wer 0.4

!head -3 data/hard_set/candidates.jsonl

## 11. Save results off the ephemeral runtime

**Do not `force_remount` while a training job is writing to Drive** — remounting
under a live writer can disrupt its checkpoint saves.

Drive was mounted read-only in section 3, so the safest option is to download
the results instead. They are small: JSON and JSONL only, no audio, no weights.

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("/content/eval-results", "zip", "results/eval")
files.download("/content/eval-results.zip")

If training is definitely **not** running on this runtime and you would rather
write straight to Drive, remount writable and copy to a separate folder —
never into `whisper-training/`:

```python
from google.colab import drive
drive.mount('/content/drive', force_remount=True)   # only when nothing is training
shutil.copytree("results/eval",
                "/content/drive/MyDrive/indic-voice-eval/latest",
                dirs_exist_ok=True)
```

## 12. Record the result

Fill in the result-entry template in `docs/EXPERIMENTS.md` with the numbers
above. Leave any field blank rather than filling it from a different run — a
partially completed entry is evidence, a plausible-looking one is not.

`results/eval/` is gitignored by default. To version the run that backs a
recorded number:

```bash
git add -f results/eval/<dir>/metrics.json results/eval/<dir>/run_config.json
```